# Tennis Court Visibility Detector

Trains an object detector (YOLO) to recognize whether a full tennis court is actually visible in a frame, using the forked `tennis-court-keypoints` dataset from Roboflow Universe. This is meant to replace the keypoint-regression-based court check, which can't say "no court here" -- it always outputs a plausible-looking (but sometimes hallucinated) set of keypoints. A detection model can simply not detect anything on frames without a court (closeups, crowd shots, graphics, replays).

Run this top to bottom in Colab with a GPU runtime (Runtime > Change runtime type > GPU).

In [ ]:
!pip install -q -U ultralytics roboflow

# Get Dataset

Replace the values below with the "Show download code" snippet from your forked Roboflow project (Dataset > Versions > Export, format = YOLOv8). Your `api_key`, workspace, project slug, and version number will all be specific to your fork.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT_SLUG")
version = project.version(1)  # replace with your fork's version number
dataset = version.download("yolov8")

# Training the model

YOLO26 nano -- small, fast on CPU for later local inference, and plenty for a visually distinct task like this. If `yolo26n.pt` isn't resolvable yet in your installed `ultralytics` version, fall back to `yolov8n.pt` (swap the `model=` value below).

In [ ]:
!yolo task=detect mode=train model=yolo26n.pt data={dataset.location}/data.yaml epochs=50 imgsz=640

# Download the trained weights

Grabs `best.pt` from the most recent training run and downloads it to your computer. Drop it into `models/` in the `tennis_analysis` repo (e.g. `models/court_detector.pt`) and load it locally with `ultralytics.YOLO(...)`, same pattern as the existing player/ball trackers.

In [ ]:
import glob
import shutil

from google.colab import files

latest_run = sorted(glob.glob("runs/detect/train*"))[-1]
weights_path = f"{latest_run}/weights/best.pt"
shutil.copy(weights_path, "court_detector.pt")
files.download("court_detector.pt")